# Hansard RAG — generation, routing, and LLM evaluation

The query-time side/online, three layers:

1. **RAG answer generation** — retrieve (hybrid, our eval winner) -> build prompt -> Groq
2. **Agentic routing** — two tools: `search_contributions` (cross-corpus questions) and `get_debate` (summarise one specific debate); a router picks per question
3. **LLM evaluation** — compare prompt variants with LLM-as-a-judge (zoomcamp Module 4 style)

Prereqs: ES up with the `hansard-chunks` index, `.env` with `GROQ_API_KEY`, imports shared functions from shared_funcs/.

In [ ]:
import json
import os
import time

import pandas as pd
from dotenv import load_dotenv
from groq import Groq
from tqdm.auto import tqdm

from shared_funcs.search import hybrid_search, find_debate, get_debate_chunks

load_dotenv()
client = Groq(api_key=os.environ["GROQ_API_KEY"])

GROQ_MODEL = "llama-3.3-70b-versatile"


def llm(prompt, model=GROQ_MODEL, temperature=0.0, max_retries=5):
    for attempt in range(max_retries):
        try:
            response = client.chat.completions.create(
                model=model,
                messages=[{"role": "user", "content": prompt}],
                temperature=temperature,
            )
            return response.choices[0].message.content.strip()
        except Exception as e:
            if "429" in str(e) or "rate" in str(e).lower():
                wait = 2 ** (attempt + 2)
                print(f"  rate limited, sleeping {wait}s")
                time.sleep(wait)
            else:
                raise
    raise RuntimeError("gave up after retries")

## Tool 1 — answer from search (cross-corpus RAG)

Two prompt variants to compare in the LLM eval:
- **V1 plain**: answer from context
- **V2 attributed**: answers must attribute claims to speaker (party) and stay within the context

In [ ]:
def build_context(results):
    blocks = []    
    for r in results:       
        speaker = r["speaker"] + (f" ({r['party']})" if r.get("party") else "")        
        blocks.append(
            f"Debate: {r['debate_title']} ({r['sitting_date']})\n"            
            f"Speaker: {speaker}\n"            
            f"Text: {r['text']}"        
            )    
    return "\n\n---\n\n".join(blocks)

PROMPT_V1_PLAIN = """You are an assistant answering questions about UK parliamentary debates.
Answer the QUESTION using only the CONTEXT below.
If the context does not contain enough information, say so.

QUESTION: {question}

CONTEXT:
{context}"""


PROMPT_V2_ATTRIBUTED = """You are an assistant answering questions about UK parliamentary debates.
Answer the QUESTION using only the CONTEXT below.

Rules:
- Attribute every claim to the speaker who made it, with their party where available,
  e.g. 'Bob Blackman (Con) argued that...'
- Where speakers disagree, present the different positions side by side
- If the context does not contain enough information, say so plainly
- End with a 'Sources' line listing the debates and dates used

QUESTION: {question}

CONTEXT:
{context}"""  

def answer_from_search(question, prompt_template=PROMPT_V2_ATTRIBUTED, k=5, filters=None):    
    results = hybrid_search(question, k=k, filters=filters)    
    prompt = prompt_template.format(question=question, context=build_context(results))    
    return llm(prompt), results 

In [ ]:
answer, sources = answer_from_search("What have MPs said about hospital repairs?")
print(answer)

## Tool 2 — summarise a specific debate

Not retrieval: find the debate by title match, pull *all* its chunks in speaking order, summarise. Top-k similarity is the wrong mechanic for 'summarise this debate' - this is why it's a separate tool.

In [ ]:
SUMMARISE_PROMPT = """You are summarising a UK parliamentary debate.

Summarise the debate below:
- Lead with what the debate was about and its outcome if stated
- Cover the main points raised, attributed to speaker (party)
- Note points of disagreement between speakers
- Keep it under 300 words

DEBATE: {title} ({date})

TRANSCRIPT:
{transcript}"""


def summarise_debate(title_query):
    found = find_debate(title_query)
    if not found:
        return f"No debate found matching '{title_query}'", []
    ext_id, title = found
    chunks = get_debate_chunks(ext_id)
    transcript = build_context(chunks)
    # Guard the context window: truncate very long debates (map-reduce is the v2 upgrade)
    if len(transcript) > 60000:
        transcript = transcript[:60000] + "\n\n[transcript truncated]"
    prompt = SUMMARISE_PROMPT.format(title=title, date=chunks[0]["sitting_date"], transcript=transcript)
    return llm(prompt), chunks

In [ ]:
summary, chunks = summarise_debate("Doncaster Royal Infirmary")
print(summary)

## Router — which tool for which question?

A single cheap LLM call classifies the question. This is the agentic step: the model decides which tool to use, then we execute it.

In [ ]:
ROUTER_PROMPT = """Classify this question about UK parliamentary debates into exactly one route:

- SEARCH: asking what was said about a topic across parliament, by any/many speakers
- DEBATE_SUMMARY: asking to summarise or explain one specific named debate
- SPEAKER: asking what one specific named MP has said (about anything or a topic)

Question: {question}

Respond with ONLY a JSON object: {{"route": "...", "debate_title": "...", "speaker_name": "...", "topic": "..."}}
Use null for fields that do not apply."""


def route_question(question):
    raw = llm(ROUTER_PROMPT.format(question=question))
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"route": "SEARCH", "topic": question}  # safe fallback


def rag(question):
    decision = route_question(question)
    route = decision.get("route", "SEARCH")

    if route == "DEBATE_SUMMARY" and decision.get("debate_title"):
        answer, sources = summarise_debate(decision["debate_title"])
    elif route == "SPEAKER" and decision.get("speaker_name"):
        # speaker filter uses the exact keyword subfield
        query = decision.get("topic") or question
        results = hybrid_search(query, k=8)
        results = [r for r in results if decision["speaker_name"].lower() in (r["speaker"] or "").lower()] or results
        prompt = PROMPT_V2_ATTRIBUTED.format(question=question, context=build_context(results))
        answer, sources = llm(prompt), results
    else:
        answer, sources = answer_from_search(question)

    return {"answer": answer, "route": route, "sources": sources}

In [ ]:
for q in [
    "What have MPs said about NHS dentistry?",
    "Summarise the Doncaster Royal Infirmary debate",
    "What has Bob Blackman been raising in parliament?",
]:
    out = rag(q)
    print(f"[{out['route']}] {q}")
    print(out["answer"][:400])
    print("=" * 80)

## LLM evaluation — prompt variants, judged by LLM

Zoomcamp Module 4 pattern: sample questions from the ground truth, answer with each prompt variant, and have an LLM judge relevance. We judge **answer vs question** (does it actually address what was asked) on a 3-level scale.

In [ ]:
JUDGE_PROMPT = """You are an expert evaluator for a question-answering system about UK parliamentary debates.

Classify how well the ANSWER addresses the QUESTION:
- RELEVANT: directly and substantively addresses the question
- PARTLY_RELEVANT: addresses the topic but incompletely or vaguely
- NON_RELEVANT: does not address the question, or only says information is missing

QUESTION: {question}

ANSWER: {answer}

Respond with ONLY a JSON object: {{"relevance": "...", "explanation": "one short sentence"}}"""


def judge(question, answer):
    raw = llm(JUDGE_PROMPT.format(question=question, answer=answer), model="llama-3.1-8b-instant")
    raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()
    try:
        return json.loads(raw)
    except json.JSONDecodeError:
        return {"relevance": "PARSE_ERROR", "explanation": raw[:100]}

In [ ]:
import csv
from pathlib import Path

N_EVAL_QUESTIONS = 50
EVAL_PATH = Path("data/processed/llm_eval.csv")

ground_truth = pd.read_csv("data/processed/ground_truth.csv")
eval_questions = ground_truth.sample(N_EVAL_QUESTIONS, random_state=42)["question"].tolist()
variants = {"v1_plain": PROMPT_V1_PLAIN, "v2_attributed": PROMPT_V2_ATTRIBUTED}

done = set()
if EVAL_PATH.exists():
    done = set(zip(*(pd.read_csv(EVAL_PATH)[["variant", "question"]].values.T.tolist())))
    print(f"resuming: {len(done)} already done")

with EVAL_PATH.open("a", newline="") as f:
    writer = csv.DictWriter(f, fieldnames=["variant", "question", "answer", "relevance", "explanation"])
    if EVAL_PATH.stat().st_size == 0:
        writer.writeheader()
    for name, template in variants.items():
        for q in tqdm(eval_questions, desc=name):
            if (name, q) in done:
                continue
            answer, _ = answer_from_search(q, prompt_template=template)
            verdict = judge(q, answer)
            writer.writerow({"variant": name, "question": q, "answer": answer,
                             "relevance": verdict.get("relevance"),
                             "explanation": verdict.get("explanation", "")[:200]})
            f.flush()
            time.sleep(2)

llm_eval = pd.read_csv(EVAL_PATH)
llm_eval.groupby("variant")["relevance"].value_counts(normalize=True).unstack().round(3)

## Read the results

- The table shows the share of RELEVANT / PARTLY_RELEVANT / NON_RELEVANT per variant - the winner goes into the app.
- Spot-check a few NON_RELEVANT rows by eye: is the answer actually bad, or is the judge wrong? Note findings in the README.
- Costs so far are all free-tier Groq; the sleep keeps runs slow but safe. Halve N_EVAL_QUESTIONS if it drags.

In [ ]:
# Eyeball the failures
llm_eval = pd.read_csv("data/processed/llm_eval.csv")
llm_eval.groupby("variant")["relevance"].value_counts(normalize=True).unstack().round(3)
llm_eval[llm_eval["relevance"] == "NON_RELEVANT"][["variant", "question", "explanation"]].head(10)